In [2]:
import numpy as np 
import pandas as pd
from dataclasses import dataclass ## for defining datatypes remove the need for defining classes with init functions ##
from typing import List, Dict ## for defining datatypes for lists and dictionaries ##


 ### defining datatypes  ###
@dataclass
class Job:
    id: int
    workload: float
 
@dataclass 
class Node:
    id: int
    speed: float
    cost_per_hour: float
    
    def calculate_time(self, job: Job):
        return job.workload / self.speed
    
    
    

In [3]:

    
def generate_cloud_environment(num_jobs=50):
    nodes = [
        Node(id=1, speed=100, cost_per_hour=0.02),
        Node(id=2, speed=250, cost_per_hour=0.08),
        Node(id=3, speed=500, cost_per_hour=0.15)
    ]

    np.random.seed(42)
    mu, sigma = 6, 1.2  ## the mean and standard deviation for workload ##

    workloads = np.random.lognormal(mean=mu, sigma=sigma, size=num_jobs)  ## generating workloads using lognormal distribution ##
    workload = np.clip(workloads, a_min=10, a_max=None) 

    jobs = [Job(id=i, workload=workload[i]) for i in range(num_jobs)] 

    return jobs, nodes 


jobs, nodes = generate_cloud_environment(num_jobs=10)

print("Nodes:")
for node in nodes:
    print(f"Node ID: {node.id}, Speed: {node.speed} MIPS, Cost per Hour: ${node.cost_per_hour}")

print("\nJobs:")    
for job in jobs:
    print(f"Job ID: {job.id}, Workload: {job.workload:.2f} MI")

Nodes:
Node ID: 1, Speed: 100 MIPS, Cost per Hour: $0.02
Node ID: 2, Speed: 250 MIPS, Cost per Hour: $0.08
Node ID: 3, Speed: 500 MIPS, Cost per Hour: $0.15

Jobs:
Job ID: 0, Workload: 732.20 MI
Job ID: 1, Workload: 341.75 MI
Job ID: 2, Workload: 877.63 MI
Job ID: 3, Workload: 2508.99 MI
Job ID: 4, Workload: 304.61 MI
Job ID: 5, Workload: 304.61 MI
Job ID: 6, Workload: 2683.98 MI
Job ID: 7, Workload: 1013.25 MI
Job ID: 8, Workload: 229.67 MI
Job ID: 9, Workload: 773.61 MI


In [4]:
def evaluate_schedule(jobs: List[Job], nodes: List[Node], assignments: List[int]) -> dict:
    """
    Takes a 1D array of assignments and returns the makespan and cost.
    Example assignments: [1, 3, 2] means:
    - job 0 is assigned to node 1
    - job 1 is assigned to node 3
    - job 2 is assigned to node 2

    Args:
        jobs: List of Job objects.
        nodes: List of Node objects.
        assignments: List of node IDs corresponding to each job's assigned node.

    Returns:
        A dictionary with schedule metrics.
    """
    # 1. Create empty queues for each node (using the Node's actual ID)
    node_queues = {node.id: [] for node in nodes}

    # 2. Route the jobs to their assigned nodes
    for job_idx, assigned_node_id in enumerate(assignments):
        current_job = jobs[job_idx]
        node_queues[assigned_node_id].append(current_job)

    total_cost = 0.0
    node_completion_times = {}

    # 3. Process each node's queue
    for node in nodes:
        queue = node_queues[node.id]

        # --- THE LOCAL HEURISTIC (Shortest Job First) ---
        # Sort the queue locally so the smallest workloads run first
        queue.sort(key=lambda j: j.workload)

        node_active_time = 0.0

        # Execute the jobs
        for job in queue:
            execution_time = job.workload / node.speed
            node_active_time += execution_time

        # Calculate cost for this specific node (Convert seconds to hours)
        node_cost = (node_active_time / 3600) * node.cost_per_hour
        total_cost += node_cost

        # Record how long this node took to finish all its work
        node_completion_times[node.id] = node_active_time

    # 4. The Makespan is the time of the node that finishes absolutely last
    makespan = max(node_completion_times.values()) if node_completion_times else 0.0

    return {
        "makespan_seconds": makespan,
        "total_cost_dollars": total_cost,
        "node_details": node_completion_times
    }

In [5]:
# Extract the valid node IDs (1, 2, and 3 from our previous cell)
available_node_ids = [n.id for n in nodes]

# Generate a random schedule (The "Chromosome")
# e.g., randomly assigning each of the 10 jobs to Node 1, 2, or 3
np.random.seed(42) # Keeping it reproducible
random_schedule = np.random.choice(available_node_ids, size=len(jobs))

print(f"Generated Random Schedule (Array):")
print(random_schedule)
print("-" * 40)

# Run the random schedule through our engine
results = evaluate_schedule(jobs, nodes, random_schedule)

print("--- Simulation Results (Random Baseline) ---")
print(f"Makespan (Total Time):   {results['makespan_seconds']:.2f} seconds")
print(f"Total Cloud Cost:        ${results['total_cost_dollars']:.4f}")
print("\nIndividual Node Completion Times:")
for node_id, time_spent in results['node_details'].items():
    print(f"  Node {node_id}: {time_spent:.2f} seconds")

Generated Random Schedule (Array):
[3 1 3 3 1 1 3 2 3 3]
----------------------------------------
--- Simulation Results (Random Baseline) ---
Makespan (Total Time):   15.61 seconds
Total Cloud Cost:        $0.0008

Individual Node Completion Times:
  Node 1: 9.51 seconds
  Node 2: 4.05 seconds
  Node 3: 15.61 seconds


In [6]:
import numpy as np

class GeneticAlgorithm:
    def __init__(self, jobs, nodes, pop_size=50, generations=100, mutation_rate=0.1):
        self.jobs = jobs
        self.nodes = nodes
        self.pop_size = pop_size
        self.generations = generations
        self.mutation_rate = mutation_rate
        
        # We need to know the valid IDs (1, 2, 3) to assign jobs to
        self.valid_node_ids = [n.id for n in nodes]
        self.num_jobs = len(jobs)

    def initialize_population(self):
        """Creates a list of random schedules (1D arrays)."""
        return [np.random.choice(self.valid_node_ids, size=self.num_jobs) for _ in range(self.pop_size)]

    def calculate_fitness(self, schedule):
        """
        Runs the schedule through your engine.
        We want to MINIMIZE score. 
        Formula: Makespan + (Cost * Scaling_Factor)
        """
        results = evaluate_schedule(self.jobs, self.nodes, schedule)
        
        # We multiply cost by 100 so it has a mathematical weight similar to seconds
        # Otherwise, the AI will ignore cost because $0.15 is mathematically "smaller" than 20 seconds.
        fitness_score = results['makespan_seconds'] + (results['total_cost_dollars'] * 100)
        return fitness_score

    def tournament_selection(self, population, fitnesses, k=3):
        """Picks 3 random schedules, returns the best one."""
        selected_indices = np.random.choice(len(population), size=k)
        best_idx = selected_indices[np.argmin([fitnesses[i] for i in selected_indices])]
        return population[best_idx]

    def crossover(self, p1, p2):
        """Cuts two parent arrays in half and swaps them to make 2 children."""
        point = np.random.randint(1, self.num_jobs - 1)
        c1 = np.concatenate([p1[:point], p2[point:]])
        c2 = np.concatenate([p2[:point], p1[point:]])
        return c1, c2

    def mutate(self, schedule):
        """Randomly takes one job and moves it to a different node."""
        if np.random.rand() < self.mutation_rate:
            mutation_point = np.random.randint(self.num_jobs)
            new_node = np.random.choice(self.valid_node_ids)
            schedule[mutation_point] = new_node
        return schedule

    def run(self):
        """The main Evolutionary Loop."""
        population = self.initialize_population()
        
        best_overall_schedule = None
        best_overall_fitness = float('inf')
        fitness_history = [] # To graph our learning later!

        print(f"Starting GA Optimization for {self.generations} generations...")
        
        for gen in range(self.generations):
            # 1. Score everyone
            fitnesses = [self.calculate_fitness(ind) for ind in population]

            # 2. Find the best in this generation
            min_idx = np.argmin(fitnesses)
            if fitnesses[min_idx] < best_overall_fitness:
                best_overall_fitness = fitnesses[min_idx]
                best_overall_schedule = population[min_idx]

            fitness_history.append(best_overall_fitness)

            # 3. Create the next generation
            new_pop = []
            
            # Elitism: Always keep the absolute best schedule so we don't lose it
            new_pop.append(best_overall_schedule)

            while len(new_pop) < self.pop_size:
                # Pick parents
                p1 = self.tournament_selection(population, fitnesses)
                p2 = self.tournament_selection(population, fitnesses)
                
                # Make children
                c1, c2 = self.crossover(p1, p2)
                
                # Mutate and add to new population
                new_pop.append(self.mutate(c1))
                if len(new_pop) < self.pop_size:
                    new_pop.append(self.mutate(c2))

            population = new_pop

        return best_overall_schedule, fitness_history

In [7]:
# Initialize our custom AI
ga = GeneticAlgorithm(jobs, nodes, pop_size=50, generations=100, mutation_rate=0.2)

# Run it!
best_ai_schedule, learning_history = ga.run()

# Evaluate the winning schedule
ai_results = evaluate_schedule(jobs, nodes, best_ai_schedule)

print("\n" + "="*40)
print("🏆 AI OPTIMIZATION COMPLETE 🏆")
print("="*40)
print(f"Final Optimized Schedule Array:\n{best_ai_schedule}\n")

print("--- Final Metrics (AI vs Random Baseline) ---")
print(f"AI Makespan: {ai_results['makespan_seconds']:.2f} sec")
print(f"AI Cost:     ${ai_results['total_cost_dollars']:.4f}")

# Look at how it balanced the load
print("\nAI Node Utilization (Notice how it balances the load):")
for node_id, time_spent in ai_results['node_details'].items():
    print(f"  Node {node_id}: {time_spent:.2f} seconds")

Starting GA Optimization for 100 generations...

🏆 AI OPTIMIZATION COMPLETE 🏆
Final Optimized Schedule Array:
[3 3 1 2 1 3 3 3 2 3]

--- Final Metrics (AI vs Random Baseline) ---
AI Makespan: 11.82 sec
AI Cost:     $0.0008

AI Node Utilization (Notice how it balances the load):
  Node 1: 11.82 seconds
  Node 2: 10.95 seconds
  Node 3: 11.70 seconds


In [8]:
def greedy_sjf_schedule(jobs: List[Job], nodes: List[Node]) -> List[int]:
    """
    A deterministic 'smart' baseline. 
    Sorts all jobs smallest to largest, then assigns each to the node 
    that will finish it the soonest based on current loads.
    """
    # 1. Sort jobs by workload (smallest first) while keeping their original index
    sorted_jobs = sorted(enumerate(jobs), key=lambda x: x[1].workload)
    
    # Empty array to hold our final answers
    assignments = [0] * len(jobs)
    
    # Track the current active time of each node
    node_current_times = {n.id: 0.0 for n in nodes}
    
    # 2. Assign each job one by one
    for original_idx, job in sorted_jobs:
        best_node_id = None
        earliest_finish_time = float('inf')
        
        # Check which node will finish this job the earliest
        for node in nodes:
            execution_time = node.calculate_time(job)
            potential_finish_time = node_current_times[node.id] + execution_time
            
            if potential_finish_time < earliest_finish_time:
                earliest_finish_time = potential_finish_time
                best_node_id = node.id
                
        # 3. Lock in the assignment and update the node's total time
        assignments[original_idx] = best_node_id
        
        # We must add the execution time to that node's running total
        best_node = next(n for n in nodes if n.id == best_node_id)
        node_current_times[best_node_id] += best_node.calculate_time(job)
        
    return assignments

In [9]:
import pandas as pd

# 1. Run the Pure SJF
sjf_schedule = greedy_sjf_schedule(jobs, nodes)
sjf_results = evaluate_schedule(jobs, nodes, sjf_schedule)

# 2. Gather data for a clean comparison table
# Note: We are using the results from the previous cells (random_schedule and ai_results)
baseline_results = evaluate_schedule(jobs, nodes, random_schedule) # Recalculating just to be safe

comparison_data = {
    "Algorithm": ["1. Random (Dumb Baseline)", "2. Pure SJF (Smart Baseline)", "3. Genetic Algorithm (AI)"],
    "Makespan (Seconds)": [
        baseline_results['makespan_seconds'], 
        sjf_results['makespan_seconds'], 
        ai_results['makespan_seconds']
    ],
    "Total Cost ($)": [
        baseline_results['total_cost_dollars'], 
        sjf_results['total_cost_dollars'], 
        ai_results['total_cost_dollars']
    ]
}

df = pd.DataFrame(comparison_data)

print("\n" + "="*60)
print(" 📊 ALGORITHM PERFORMANCE COMPARISON 📊 ")
print("="*60)
# We use .to_string(index=False) to hide the dataframe row numbers for a cleaner look
print(df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("="*60)


 📊 ALGORITHM PERFORMANCE COMPARISON 📊 
                   Algorithm  Makespan (Seconds)  Total Cost ($)
   1. Random (Dumb Baseline)             15.6122          0.0008
2. Pure SJF (Smart Baseline)             15.0488          0.0008
   3. Genetic Algorithm (AI)             11.8224          0.0008
